Sentiment Analysis Hyperparameter Tuning

This notebook is meant to compare Optuna hyperparameter tuning with GridSearchCV hyperparameter tuning with some of the hyperparameters of a sentiment analysis model that uses logistic regression and TF-IDF vectorization.

I will start with Optuna hyperparameter tuning

In [16]:
# Import all relevant modules
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold

Data extraction, quick visualization, and train-test-split preparation

In [2]:
data = pd.read_csv('sentiment_analysis.csv')
data.head()

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           499 non-null    int64 
 1   Month          499 non-null    int64 
 2   Day            499 non-null    int64 
 3   Time of Tweet  499 non-null    object
 4   text           499 non-null    object
 5   sentiment      499 non-null    object
 6   Platform       499 non-null    object
dtypes: int64(3), object(4)
memory usage: 27.4+ KB


In [4]:
X = data["text"]
y = data["sentiment"]

In [5]:
data['sentiment'].value_counts(normalize=True)

sentiment
neutral     0.398798
positive    0.332665
negative    0.268537
Name: proportion, dtype: float64

In [38]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, stratify=y_train_full, test_size=0.25, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

Training set size: 299
Validation set size: 100
Test set size: 100


In [39]:
import optuna

def objective(trial):
    # Suggest hyperparameters
    ngram_range_str = trial.suggest_categorical('ngram_range', ['1-1', '1-2', '1-3', '2-2'])
    ngram_range = tuple(map(int, ngram_range_str.split('-')))
    max_df = trial.suggest_float('max_df', 0.6, 1.0)
    min_df = trial.suggest_int('min_df', 1, 10)
    C = trial.suggest_float('C', 1e-3, 10.0, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'lbfgs'])

    # Pipeline: TF-IDF + Logistic Regression
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram_range, max_df=max_df, min_df=min_df)),
        ('clf', LogisticRegression(C=C, solver=solver, max_iter=1000))
    ])

    # Fit on training set
    pipeline.fit(X_train, y_train)

    # Predict on validation set
    preds = pipeline.predict(X_val)
    acc = accuracy_score(y_val, preds)

    return acc  # maximize accuracy

In [40]:
import logging
logging.getLogger('optuna').setLevel(logging.WARNING)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [41]:
print("Best trial:")
trial = study.best_trial

print(f"  Accuracy: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Accuracy: 0.7100
  Params: 
    ngram_range: 1-3
    max_df: 0.8775057375629275
    min_df: 1
    C: 5.417363001942155
    solver: lbfgs


In [43]:
# Get the best parameters from the study
best_params = study.best_trial.params

ngram_range_str = best_params['ngram_range']
best_params['ngram_range'] = tuple(map(int, ngram_range_str.split('-')))

final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=best_params['ngram_range'],
        max_df=best_params['max_df'],
        min_df=best_params['min_df']
    )),
    ('clf', LogisticRegression(
        C=best_params['C'],
        solver=best_params['solver'],
        max_iter=1000
    ))
])

final_pipeline.fit(X_train_full, y_train_full)

y_pred = final_pipeline.predict(X_test)

print("Classification Report for Final Model:")
print(classification_report(y_test, y_pred))

test_acc = final_pipeline.score(X_test, y_test)
print(f"Final Test Accuracy: {test_acc:.4f}")

Classification Report for Final Model:
              precision    recall  f1-score   support

    negative       0.85      0.41      0.55        27
     neutral       0.61      0.88      0.72        40
    positive       0.77      0.70      0.73        33

    accuracy                           0.69       100
   macro avg       0.74      0.66      0.67       100
weighted avg       0.73      0.69      0.68       100

Final Test Accuracy: 0.6900


The final test accuracy is 0.69. Now we will use GridSearchCV.

In [11]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression())
])

In [12]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', LogisticRegression())])

In [44]:
params = {
    'tfidf__ngram_range': [(1,1), (1,2), (1,3), (2,2)],
    'tfidf__max_features': [None, 1000, 5000, 10000],
    'tfidf__stop_words': [None, 'english'],
    'tfidf__min_df': [1, 2],
    'tfidf__max_df': [0.9, 1.0]
}

In [45]:
grid = GridSearchCV(pipeline, param_grid=params, cv=3, scoring='accuracy', verbose=2, n_jobs=1)

In [46]:
grid.fit(X_train_full, y_train_full)

Fitting 3 folds for each of 128 candidates, totalling 384 fits
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__stop_words=None; total time=   0.0s
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__stop_words=None; total time=   0.0s
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__stop_words=None; total time=   0.0s
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__stop_words=english; total time=   0.0s
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__stop_words=english; total time=   0.0s
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngram_range=(1, 1), tfidf__stop_words=english; total time=   0.0s
[CV] END tfidf__max_df=0.9, tfidf__max_features=None, tfidf__min_df=1, tfidf__ngra

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('clf', LogisticRegression())]),
             n_jobs=1,
             param_grid={'tfidf__max_df': [0.9, 1.0],
                         'tfidf__max_features': [None, 1000, 5000, 10000],
                         'tfidf__min_df': [1, 2],
                         'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 2)],
                         'tfidf__stop_words': [None, 'english']},
             scoring='accuracy', verbose=2)

In [47]:
print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best parameters: {'tfidf__max_df': 0.9, 'tfidf__max_features': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__stop_words': 'english'}
Best CV score: 0.6340852130325815


In [48]:
y_pred_grid = grid.predict(X_test)

print("Classification Report for GridSearchCV Model:")
print(classification_report(y_test, y_pred_grid))

test_acc_grid = accuracy_score(y_test, y_pred_grid)
print(f"Final Test Accuracy from GridSearchCV: {test_acc_grid:.4f}")

Classification Report for GridSearchCV Model:
              precision    recall  f1-score   support

    negative       0.88      0.26      0.40        27
     neutral       0.59      0.95      0.73        40
    positive       0.86      0.73      0.79        33

    accuracy                           0.69       100
   macro avg       0.78      0.65      0.64       100
weighted avg       0.76      0.69      0.66       100

Final Test Accuracy from GridSearchCV: 0.6900


In [49]:
from sklearn.metrics import classification_report
import pandas as pd

# --- Create the Summary Table ---

# 1. Get the classification reports as dictionaries
optuna_report = classification_report(y_test, y_pred, output_dict=True)
grid_report = classification_report(y_test, y_pred_grid, output_dict=True)

# 2. Create a dictionary to hold the final comparison data
comparison_data = {
    "Metric": ["Accuracy", "F1 Score (Macro Avg)", "Precision (Macro Avg)", "Recall (Macro Avg)"],
    "Optuna": [
        optuna_report['accuracy'],
        optuna_report['macro avg']['f1-score'],
        optuna_report['macro avg']['precision'],
        optuna_report['macro avg']['recall']
    ],
    "GridSearchCV": [
        grid_report['accuracy'],
        grid_report['macro avg']['f1-score'],
        grid_report['macro avg']['precision'],
        grid_report['macro avg']['recall']
    ]
}

# 3. Create and display the DataFrame
comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index("Metric")

# 4. Format the numbers to 4 decimal places for readability
pd.options.display.float_format = '{:.4f}'.format

print("--- Final Model Comparison ---")
display(comparison_df)

--- Final Model Comparison ---


,Optuna,GridSearchCV
Metric,,
Accuracy,0.6900,0.6900
F1 Score (Macro Avg),0.6673,0.6392
Precision (Macro Avg),0.7423,0.7753
Recall (Macro Avg),0.6598,0.6455
